# ALL_RETRIEVER

# HyDE (Hypothetical Document Embeddings)

## What is HyDE?

Generate a **fake answer** to the query first, then search using that answer instead of the original question.

**Why?** Answers match documents better than questions do.

---

## How It Works

In [ ]:
**Traditional RAG:**
Query → Embed → Search → Docs → Answer

In [ ]:
**HyDE:**
Query → LLM (fake answer) → Embed fake answer → Search → Docs → Real Answer

---

## Example

**Query:** "What causes inflation?"

In [ ]:
**HyDE Step 1 - Generate hypothesis:**
"Inflation is caused by increased money supply, rising demand, 
and supply chain disruptions."

**HyDE Step 2 - Search with this answer**

→ Finds documents about inflation causes (better match than searching with the question!)

---

## Code Example

In [ ]:
# 1. Generate hypothetical answer
hyde_prompt = "Write a detailed answer: {query}"
hypothetical_answer = llm.invoke(hyde_prompt.format(query=query))

# 2. Search using the hypothetical answer
docs = vectorstore.similarity_search(hypothetical_answer, k=5)

# 3. Generate final answer
final_answer = llm.invoke(f"Context: {docs}\nQuestion: {query}")

---

## When to Use

| Use HyDE ✅ | Skip HyDE ❌ |

|-------------|--------------|

| Complex/technical queries | Simple keyword lookups |

| Poor retrieval quality | Good enough accuracy |

| Accuracy > Speed | Need low latency |

| Domain-specific content | Budget-constrained (2x cost) |

**Impact**: +10-15% accuracy, +500ms latency, 2x LLM costs

---

## Key Takeaway

HyDE bridges the semantic gap between **questions** (queries) and **answers** (documents) by searching with a hypothetical answer instead of the raw question.

**Trade-off**: Better retrieval at the cost of extra LLM call.

---

# Advanced RAG Retrieval Methods - Quick Reference

## When to Use Which Retrieval Method?

| Method | Use When | Avoid When | Key Benefit |

|--------|----------|------------|-------------|

| **HyDE** | Complex queries, poor retrieval | Simple lookups, low latency needed | Bridges question-answer gap (+10-15% accuracy) |

| **SelfQuery** | Rich metadata, filter terms in query | No metadata, budget-constrained | Natural language filters (no syntax needed) |

| **CRAG** | Variable retrieval quality, time-sensitive | Good retrieval, latency-critical | Self-corrects with web search fallback |

| **Parent Document** | Complex docs, need full context | Short docs, memory-constrained | Precise search + complete context |

| **Ensemble** | Production RAG, need weighted fusion | Prototyping, simplicity preferred | Weighted RRF for smart re-ranking |

### Quick Decision:

- **Need metadata filtering?** → SelfQueryRetriever

- **Retrieval sometimes fails?** → CRAG  

- **Questions don't match docs well?** → HyDE

- **Chunks too small, missing context?** → Parent Document

- **Want keyword + semantic search?** → Ensemble Retriever (weighted RRF)

---

# SelfQueryRetriever

## What is it?

`SelfQueryRetriever` does **both** — it uses an LLM to split the natural language query into two parts:

1. **Semantic search query** → embedded and used for ANN vector search

2. **Structured filter** → extracted from metadata fields, applied as a pre-filter before ANN runs

**Purpose**: Users can include filters in natural language without knowing filter syntax — and the semantic search still runs on the filtered subset, not the full corpus.

---

## How It Works

In [ ]:
User query: "Show me finance policies from 2023 about expense limits"
                        │
               LLM (query constructor)
                        │
          ┌─────────────┴─────────────┐
     semantic_query              structured_filter
    "expense limits"         dept=finance, year>=2023
          │                          │
    embed + ANN search          pre-filter on metadata
    over filtered subset              │
          └─────────────┬─────────────┘
                   Top-k results

The LLM produces a structured object with **two fields**:

In [ ]:
{
  "query": "expense limits",        ← this goes to ANN vector search
  "filter": {
    "department": {"$eq": "finance"},
    "doc_type":   {"$eq": "policy"},
    "year":       {"$gte": 2023}
  }
}

**Important**: `metadata_field_info` only tells the LLM which fields exist and what they mean — it has no effect on the vector search. If the query has no filterable intent, the LLM returns `filter: None` and the full query goes straight to ANN search.

---

## Code Example

In [ ]:
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo

# Define filterable metadata fields
metadata_fields = [
    AttributeInfo(name="genre", description="Movie genre", type="string"),
    AttributeInfo(name="year", description="Release year", type="integer"),
    AttributeInfo(name="rating", description="IMDB rating", type="float"),
]

# Create retriever
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Movie descriptions",
    metadata_field_info=metadata_fields
)

# Query with filterable intent — LLM splits into semantic_query + filter
results = retriever.invoke("Action movies from 2023 with rating above 7")
# semantic_query → "action movies"   (ANN search over filtered subset)
# filter         → {genre: "action", year: 2023, rating: {$gt: 7}}

# Query with no filterable intent — full query goes to vector search, no filter
results = retriever.invoke("What makes a good movie?")
# semantic_query → "What makes a good movie?"  (ANN over all docs)
# filter         → None

---

## Pros & Cons

| ✅ Pros | ❌ Cons |

|---------|---------|

| Natural language filters (no syntax needed) | Extra LLM call (+cost, +latency) |

| Both semantic + exact metadata filtering | Requires structured metadata |

| User-friendly ("recent" → date filter) | 5-10% LLM parsing errors |

| Handles complex multi-filter queries | More setup complexity |

**Impact**: Query "cheap laptops under $500"

- Without: Returns all laptop content

- With: Only laptops where `price < 500` ✅

---

## When to Use

| ✅ Use When | ❌ Skip When |

|-------------|--------------|

| Documents have rich metadata | No useful metadata |

| Queries include filters ("recent", "cheap") | Simple content-only queries |

| Precision matters (exact field matching) | Latency-critical |

| Non-technical users | Budget-constrained |

**Good Use Cases:**

- **E-commerce**: "wireless headphones under $100"

- **Research**: "AI papers from 2024"

- **Movies**: "sci-fi from the 80s"

- **Legal**: "California cases after 2020"

---

## Comparison

| Aspect | Traditional RAG | SelfQueryRetriever |

|--------|----------------|-------------------|

| **Query** | Manual filter syntax | Natural language auto-parsed |

| **Search scope** | All vectors | Pre-filtered subset |

| **Precision** | Semantic only | Semantic + exact metadata |

| **Latency** | ~100ms | ~400ms (+LLM call) |

| **Cost** | Embedding only | Embedding + LLM |

| **UX** | Must know syntax | Natural language |

---

## Example Queries

| Natural Language | semantic_query | Auto-Extracted Filter |

|------------------|---------------|-----------------------|

| "Recent AI articles" | "AI articles" | `{date: {$gte: recent}}` |

| "Cheap Paris hotels" | "hotels" | `{location: "Paris", price: {$lt: low}}` |

| "Python for beginners" | "Python tutorial" | `{level: "beginner"}` |

| "90s action movies" | "action movies" | `{genre: "action", year: {$gte: 1990, $lte: 2000}}` |

---

## Key Takeaway

`SelfQueryRetriever` = **LLM query decomposition** → **pre-filtered ANN search**

It always runs **both** vector search and metadata filtering together — not just filtering alone.

**Trade-off**: Better precision + UX at cost of extra LLM call.

**Best for**: Rich metadata + users who include filtering criteria naturally in queries.

---

# Corrective RAG (CRAG)

## What is it?

**Self-corrects** retrieval by evaluating document relevance and falling back to web search if needed.

**Purpose**: Improve answer quality when retrieved docs are low-quality or irrelevant.

---

## How It Works

In [ ]:
Query → Retrieve Docs
    ↓
Evaluate relevance (LLM grader)
    ↓
┌─────────┬──────────┬─────────┐
│ Good    │ Partial  │ Bad     │
├─────────┼──────────┼─────────┤
│ Use     │ Filter + │ Discard │
│ as-is   │ Web      │ → Web   │
└─────────┴──────────┴─────────┘
    ↓
Generate Answer

---

## Code Example

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainFilter

# Grade retrieved documents
def grade_documents(docs, query):
    grader_prompt = "Does this doc help answer '{query}'? Yes/No"
    grader = LLMChainFilter.from_llm(llm, prompt=grader_prompt)
    return grader.compress_documents(docs, query)

# CRAG pipeline
docs = retriever.get_relevant_documents(query)
graded_docs = grade_documents(docs, query)

if not graded_docs:  # All docs irrelevant
    web_docs = web_search(query)
    final_docs = web_docs
else:
    final_docs = graded_docs

answer = llm.invoke(f"Context: {final_docs}\nQ: {query}")

---

## Pros & Cons

| ✅ Pros | ❌ Cons |

|---------|---------|

| Self-corrects bad retrieval | 2-3x more LLM calls |

| Falls back to fresh web data | Slower (~2-3s extra) |

| Higher answer accuracy | Complex pipeline |

| Handles edge cases | Higher cost |

**Impact**: Query "Latest iPhone price"

- Regular RAG: Uses outdated docs → wrong price

- CRAG: Detects outdated → web search → correct price ✅

---

## When to Use

| ✅ Use | ❌ Skip |

|--------|---------|

| Retrieval quality varies | Consistent good retrieval |

| Need high accuracy | Budget/latency constrained |

| Time-sensitive queries | Static knowledge base |

| External validation needed | Simple queries |

---

## Key Takeaway

**CRAG** = Built-in quality control that auto-corrects bad retrieval with web search

**Trade-off**: Better accuracy at cost of 2-3x LLM calls + latency

---

# Parent Document Retrieval

## What is it?

Search with **small chunks** (precise matching), but return **full parent documents** (complete context).

**Purpose**: Get best of both - precise retrieval + complete context for LLM.

---

## How It Works

In [ ]:
Document: "AI Tutorial - Part 1, 2, 3"
    ↓
Split into small chunks
    ↓
Chunk 1: "Part 1: Intro"  ←  Search finds this
Chunk 2: "Part 2: Details"
Chunk 3: "Part 3: Advanced"
    ↓
Return: FULL parent doc (Parts 1+2+3)

---

## Storage Architecture (TWO Separate Stores!)

**Important:** Parent and child chunks use **different storage systems**:

In [ ]:
┌─────────────────────────────────────────┐
│ VectorDB (Searchable)                   │
│ - Child chunks (embedded)               │
│ - Metadata: {parent_id: "doc_123"}     │
└─────────────────────────────────────────┘
         │
         │ (1) Search finds child
         │ (2) Get parent_id from metadata
         ↓
┌─────────────────────────────────────────┐
│ DocStore (Key-Value Storage)            │
│ - Parent docs (NOT embedded)            │
│ - Key: "doc_123" → Full parent text    │
└─────────────────────────────────────────┘

**How retriever identifies parent vs child:**

1. Only **child chunks are embedded** and stored in VectorDB

2. Each child has `metadata.parent_id` linking to parent

3. **Parent docs stored in separate DocStore** (InMemoryStore)

4. Retriever searches child → extracts parent_id → fetches from DocStore

**How ParentDocumentRetriever Links Children to Parents:**

When you call `retriever.add_documents()`, the retriever automatically:

1. Splits documents into parent chunks (using `parent_splitter`)

2. Splits each parent into child chunks (using `child_splitter`)

3. **Adds `parent_id` metadata** to each child chunk (auto-generated UUID)

4. Stores parent chunks in `docstore` (keyed by `parent_id`)

5. Stores child chunks in `vectorstore` with embeddings + `parent_id` metadata

In [ ]:
**Retrieval Flow:**
Query → Search child embeddings → Extract parent_ids from matched children → Fetch full parents from docstore → Return parent documents

---

## Code Example

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Parent splitter (large chunks)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)

# Child splitter (small chunks for search)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

# SEPARATE storage for parents (not embedded!)
store = InMemoryStore()

# Create retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,  # For child chunks (embedded)
    docstore=store,           # For parent docs (NOT embedded)
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Add documents (auto-creates parent-child relationships)
retriever.add_documents(documents)

# Search with small chunks, get full parents
docs = retriever.get_relevant_documents("machine learning basics")
# 1. VectorDB finds matching child chunks
# 2. Extracts parent_id from child metadata
# 3. Fetches full parent from DocStore
# Returns complete parent docs, not just matching chunks

---

## Storage Details

| Component | Storage | Embedded? | Purpose |

|-----------|---------|-----------|---------|

| **Child chunks** | VectorDB | ✅ Yes | Searchable (precise matching) |

| **Parent docs** | DocStore | ❌ No | Retrieved (complete context) |

| **Metadata** | VectorDB | N/A | Links child → parent_id |

**Why separate storage?**

- Child chunks need embeddings for search

- Parent docs don't need embeddings (just stored for retrieval)

- Saves embedding costs and storage space

---

## Pros & Cons

| ✅ Pros | ❌ Cons |

|---------|---------|

| Precise search (small chunks) | More storage (2 stores) |

| Complete context (full docs) | Extra complexity |

| No info loss | Slower indexing |

| Better LLM answers | May include irrelevant parts |

**Impact**: Query "What is backpropagation?"

- Regular RAG: Returns 400-char chunk → incomplete explanation

- Parent Retrieval: Returns full section → complete explanation ✅

---

## When to Use

| ✅ Use | ❌ Skip |

|--------|---------|

| Complex documents | Simple/short docs |

| Need full context | Memory constrained |

| Multi-part content | Standalone chunks work |

| Technical explanations | Quick lookups |

**Best for**: Technical docs, tutorials, long-form content where context matters

---

## Key Takeaway

**Parent Retrieval** = Search small (precision) + Return large (context)

**Architecture**: Child chunks in VectorDB (embedded) + Parent docs in DocStore (not embedded)

**Trade-off**: Better context at cost of 2 storage systems + complexity

---

# Ensemble Retriever vs Merger Retriever

## What are they?

Both combine multiple retrievers (BM25, semantic, multi-query), but with different strategies:

- **Ensemble**: Weighted + Reciprocal Rank Fusion (RRF)

- **Merger**: Simple concatenation

---

## Key Differences

| Feature | Ensemble Retriever | Merger Retriever |

|---------|-------------------|------------------|

| **Merging** | Weighted RRF | Simple concat |

| **Weights** | ✅ Yes (0.6, 0.4) | ❌ No |

| **Re-ranking** | ✅ Smart fusion | ❌ None |

| **Speed** | Slower | Faster |

| **Best for** | Production RAG | Quick prototyping |

---

## How They Work

In [ ]:
**Ensemble (Recommended for RAG):**
Query → ┌─ Vector (60%) → Rank 1,2,3
        └─ BM25 (40%) → Rank 1,2,3
            ↓
        RRF Weighted Fusion
            ↓
        Smart Re-ranked Results

In [ ]:
**Merger (Simple):**
Query → ┌─ Vector → Docs A
        └─ BM25 → Docs B
            ↓
        Concat A + B
            ↓
        Deduplicate

---

## Code Examples

### Ensemble Retriever (Weighted + RRF)

In [ ]:
from langchain.retrievers import EnsembleRetriever, BM25Retriever

# Create retrievers
bm25_retriever = BM25Retriever.from_documents(docs)
vector_retriever = vectorstore.as_retriever()

# Ensemble with weights (trust semantic more)
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.6, 0.4]  # 60% semantic, 40% keyword
)

# Uses RRF to intelligently merge results
results = ensemble_retriever.get_relevant_documents("machine learning")

### Merger Retriever (Simple Merge)

In [ ]:
from langchain.retrievers import MergerRetriever

# Merger - no weights, just combines
merger_retriever = MergerRetriever(retrievers=[
    vector_retriever,
    bm25_retriever
])

# Simple concatenation + deduplication
results = merger_retriever.get_relevant_documents("machine learning")

---

## When to Use Which?

| ✅ Use Ensemble | ✅ Use Merger |

|-----------------|---------------|

| Production RAG systems | Quick prototyping |

| Need weighted control | Don't care about weights |

| Want smart re-ranking | Need fastest merge |

| Accuracy matters | Simplicity preferred |

---

## Impact Example

**Query:** "neural networks"

**Merger:**

- Gets: Vec[1,2,3] + BM25[1,2,3] → Concat → 6 docs

- Issue: No ranking intelligence

**Ensemble:**

- Gets: Vec[1,2,3] (0.6) + BM25[1,2,3] (0.4) → RRF → 5 docs

- Better: Smart weighted fusion ✅

---

## Key Takeaway

**Ensemble (Recommended)**: Weighted RRF for production RAG  

**Merger**: Quick simple merge for prototyping

**Trade-off**: Ensemble = Better accuracy vs Merger = Simpler/faster